# Lab 01 — Fundamentos de Apache Spark e PySpark

**Certificação:** AWS Certified Data Engineer – Associate (DEA-C01)  
**Domínio:** 1 — Ingestão e transformação de dados  

## O que vamos aprender

- O que é o Apache Spark e por que ele é relevante para o exame
- Como criar uma `SparkSession`
- A diferença entre transformações e ações (*lazy evaluation*)
- Como ler, transformar e escrever dados com DataFrames
- Spark SQL — usando SQL padrão sobre DataFrames

---

## Contexto AWS

O **AWS Glue** executa jobs Spark gerenciados na nuvem. Tudo que você aprender aqui se aplica diretamente a jobs Glue — a diferença principal é que o Glue adiciona o conceito de `DynamicFrame` (uma abstração sobre o `DataFrame`) e gerencia o cluster automaticamente.

> Execute as células em ordem usando **Shift+Enter**.

## Parte 1 — SparkSession

A `SparkSession` é o ponto de entrada de qualquer aplicação Spark. Ela:
- Cria o **SparkContext** (conexão com o cluster)
- Gerencia configurações da aplicação
- Fornece a API para leitura de dados

Em modo `local[*]`, o Spark usa todos os cores da máquina como se fosse um cluster de um nó. Em produção (AWS Glue, EMR), o `master` aponta para o cluster real.

In [ ]:
import os

# Configura o Java 17 instalado via Homebrew
os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType
)

spark = (
    SparkSession.builder
    .appName("Lab01 - Fundamentos Spark")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark versão : {spark.version}")
print(f"Master       : {spark.sparkContext.master}")
print("SparkSession criada com sucesso! ✅")

## Parte 2 — Leitura de dados

O Spark pode ler dados de diversas fontes: CSV, JSON, Parquet, Avro, JDBC, S3, e outras.

### Schema inferido vs schema explícito

| | Schema inferido | Schema explícito |
|---|---|---|
| Código | Menos código | Mais código |
| Performance | Faz um scan extra nos dados | Não precisa do scan extra |
| Segurança | Pode inferir tipos errados | Você controla os tipos |
| **Recomendação** | Exploração / desenvolvimento | **Produção** |

In [ ]:
# Caminho relativo à raiz do repositório
# Execute o notebook a partir da raiz: jupyter lab
DATASET_PATH = "../../../datasets/lab-01/vendas.csv"

# Schema explícito — prática recomendada em produção
schema = StructType([
    StructField("id_venda",       IntegerType(), nullable=False),
    StructField("data",           StringType(),  nullable=False),
    StructField("produto",        StringType(),  nullable=False),
    StructField("categoria",      StringType(),  nullable=False),
    StructField("quantidade",     IntegerType(), nullable=False),
    StructField("preco_unitario", DoubleType(),  nullable=False),
    StructField("regiao",         StringType(),  nullable=False),
    StructField("vendedor",       StringType(),  nullable=False),
])

df = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv(DATASET_PATH)
)

print("Schema do DataFrame:")
df.printSchema()
print(f"Total de registros: {df.count()}")

In [ ]:
# Visualizar os primeiros registros
df.show(5, truncate=False)

## Parte 3 — Lazy Evaluation

Este é um dos conceitos mais importantes do Spark. **Transformações** (como `filter`, `select`, `withColumn`) **não executam imediatamente** — elas constroem um **plano de execução** chamado DAG (*Directed Acyclic Graph*).

Só quando uma **ação** é chamada (`show`, `count`, `write`, `collect`) o Spark:
1. Analisa o DAG
2. Otimiza o plano via *Catalyst Optimizer*
3. Executa o job

**Por que isso importa?** O otimizador pode reordenar operações, eliminar colunas desnecessárias (*predicate pushdown*) e combinar etapas — resultando em execuções muito mais eficientes.

In [ ]:
# Estas linhas NÃO executam nada — apenas definem o plano
df_transformado = (
    df
    .withColumn("valor_total", F.col("quantidade") * F.col("preco_unitario"))
    .withColumn("data", F.to_date(F.col("data"), "yyyy-MM-dd"))
    .withColumn("mes", F.month(F.col("data")))
    .filter(F.col("valor_total") > 1000)
    .select("id_venda", "data", "mes", "produto", "categoria",
            "regiao", "vendedor", "quantidade", "preco_unitario", "valor_total")
)

# explain() mostra o plano SEM executar — observe as otimizações
print("Plano de execução lógico:")
df_transformado.explain(mode="simple")

In [ ]:
# AÇÃO — agora o Spark executa o plano completo
print("Resultado após transformações (valor_total > R$ 1.000):")
df_transformado.show(10, truncate=False)

## Parte 4 — Agregações

O `groupBy` + `agg` é o equivalente Spark do `GROUP BY` do SQL. Diferente do pandas, o Spark distribui as agregações pelos executores do cluster — fundamental para trabalhar com terabytes de dados.

In [ ]:
# Vendas por categoria
print("--- Receita por categoria ---")
(
    df_transformado
    .groupBy("categoria")
    .agg(
        F.count("id_venda").alias("qtd_vendas"),
        F.round(F.sum("valor_total"), 2).alias("receita_total"),
        F.round(F.avg("valor_total"), 2).alias("ticket_medio"),
    )
    .orderBy(F.desc("receita_total"))
    .show(truncate=False)
)

In [ ]:
# Top 3 vendedores
print("--- Top 3 vendedores ---")
(
    df_transformado
    .groupBy("vendedor")
    .agg(F.round(F.sum("valor_total"), 2).alias("receita_total"))
    .orderBy(F.desc("receita_total"))
    .limit(3)
    .show(truncate=False)
)

In [ ]:
# Evolução mensal
print("--- Receita mensal ---")
(
    df_transformado
    .groupBy("mes")
    .agg(F.round(F.sum("valor_total"), 2).alias("receita_mensal"))
    .orderBy("mes")
    .show(truncate=False)
)

## Parte 5 — Escrita em Parquet

**Parquet** é o formato padrão em pipelines de dados modernos e é o formato preferido no ecossistema AWS:

- **Columnar**: lê apenas as colunas necessárias — muito mais eficiente para analytics
- **Compressão**: reduz custo de armazenamento em S3 em até 75% comparado ao CSV
- **Suporte nativo**: AWS Glue, Athena, Redshift Spectrum e EMR leem Parquet nativamente

O **particionamento** divide os dados em subdiretórios por uma coluna. O Athena e o Glue usam o particionamento para fazer *partition pruning* — lendo apenas as partições relevantes para uma query.

In [ ]:
OUTPUT_PATH = "../data/output"

(
    df_transformado
    .write
    .mode("overwrite")
    .partitionBy("categoria")   # cria subdiretórios: categoria=Eletronicos/, etc.
    .parquet(OUTPUT_PATH)
)

print(f"Dados escritos em: {OUTPUT_PATH}")

# Lê de volta para validar
df_parquet = spark.read.parquet(OUTPUT_PATH)
print(f"Registros lidos do Parquet: {df_parquet.count()}")
df_parquet.show(5, truncate=False)

## Parte 6 — Spark SQL

O Spark permite usar SQL padrão registrando DataFrames como **views temporárias**. Isso é útil quando:
- Você já conhece SQL e quer expressar lógica complexa de forma familiar
- Está migrando queries existentes para um pipeline Spark
- Trabalha com analistas que não conhecem a API Python do Spark

No AWS Glue, o Spark SQL é amplamente usado em *Visual ETL* e em notebooks interativos.

In [ ]:
# Registra o DataFrame como view temporária
df_transformado.createOrReplaceTempView("vendas")

resultado_sql = spark.sql("""
    SELECT
        regiao,
        COUNT(id_venda)               AS qtd_vendas,
        ROUND(SUM(valor_total), 2)    AS receita_total
    FROM vendas
    GROUP BY regiao
    ORDER BY receita_total DESC
""")

print("--- Receita por região (Spark SQL) ---")
resultado_sql.show(truncate=False)

## Encerramento

Sempre encerre a SparkSession ao final do trabalho para liberar recursos.

In [ ]:
spark.stop()
print("✅ Lab 01 concluído!")
print()
print("Conceitos praticados:")
print("  ✓ SparkSession e configuração local")
print("  ✓ Leitura de CSV com schema explícito")
print("  ✓ Lazy evaluation e DAG de execução")
print("  ✓ Transformações: withColumn, filter, select")
print("  ✓ Agregações: groupBy, agg, sum, avg, count")
print("  ✓ Escrita em Parquet com particionamento")
print("  ✓ Spark SQL sobre views temporárias")
print()
print("Próximo: Lab 02 — AWS Glue com DynamicFrames")

---

## Resumo dos conceitos

| Conceito | Analogia | Quando usar |
|---|---|---|
| `SparkSession` | Conexão com o banco | Início de toda aplicação |
| `DataFrame` | Tabela distribuída | Dados estruturados |
| Transformação | Receita de bolo | Sempre que possível — são lazy |
| Ação | Assar o bolo | Minimize o número de ações |
| Parquet | Formato otimizado | Armazenamento e analytics |
| Spark SQL | SQL padrão | Lógica complexa ou migração |

## Links úteis

- [Documentação PySpark](https://spark.apache.org/docs/latest/api/python/)
- [AWS Glue — PySpark Extensions](https://docs.aws.amazon.com/glue/latest/dg/aws-glue-programming-python.html)
- [Otimização de jobs Spark no Glue](https://docs.aws.amazon.com/prescriptive-guidance/latest/tuning-aws-glue-for-apache-spark/key-topics-apache-spark.html)